# Chapter 10 — The Safe but Useless Model

**Book alignment:** Hallucination From First Principles, Chapter 10

**Question this notebook isolates:** Does static scoring reward vagueness while paired intervention exposes generic collapse?

All responders below are deterministic synthetic fixtures; they demonstrate the mechanism type, not real model behavior.

In [ ]:
import random
import numpy as np

random.seed(10)
rng = np.random.default_rng(10)

print("seed fixed:", 10)

## Static scoring rewards vagueness

A static rubric over fluency, plausibility, and absence of explicit error cannot falsify a balanced generality, so the generic candidate outscores a specific, falsifiable, correct one. Decision specificity reverses the ranking.

In [ ]:
def static_score(fluency, plausibility, falsifiable_burden):
    # no explicit error in either candidate; vagueness lowers the falsifiable burden
    return round(0.4 * fluency + 0.4 * plausibility + 0.2 * (1.0 - falsifiable_burden), 4)

SPECIFIC = {"text": "Cut discretionary R&D immediately and preserve twelve months of payroll.",
            "fluency": 0.90, "plausibility": 0.85, "burden": 0.90, "specificity": 0.95}
GENERIC = {"text": "Balance near-term efficiency with long-term innovation.",
           "fluency": 0.97, "plausibility": 0.95, "burden": 0.10, "specificity": 0.15}

s_specific = static_score(SPECIFIC["fluency"], SPECIFIC["plausibility"], SPECIFIC["burden"])
s_generic = static_score(GENERIC["fluency"], GENERIC["plausibility"], GENERIC["burden"])

print(f"static  specific={s_specific:.3f} generic={s_generic:.3f}")
print(f"specif. specific={SPECIFIC['specificity']:.2f} generic={GENERIC['specificity']:.2f}")

In [ ]:
assert s_generic > s_specific, (s_generic, s_specific)
assert SPECIFIC["specificity"] > GENERIC["specificity"]
print("PASS: static scoring selects the vague answer; specificity exposes the trade")

## Paired inversion: acknowledgment is not uptake

Reverse the decisive fact (runway 18 months -> 3 months). Mentioning the fact is acknowledgment; moving the extracted decision in the oracle-required direction is uptake. Cosmetic adaptation passes the first gate and fails the second.

In [ ]:
ORACLE = {"growth": "INVEST_FOR_GROWTH", "distress": "PRESERVE_CASH"}

RESPONSIVE = {"growth": ("INVEST_FOR_GROWTH", True), "distress": ("PRESERVE_CASH", True)}
# generic stub mentions the runway in both cases, then recommends the same hybrid anyway
COLLAPSED = {"growth": ("HYBRID", True), "distress": ("HYBRID", True)}


def gates(trace):
    d_g, ack_g = trace["growth"]
    d_d, ack_d = trace["distress"]
    ack = bool(ack_g and ack_d)
    uptake = bool(d_g != d_d)
    fidelity = bool(d_g == ORACLE["growth"] and d_d == ORACLE["distress"])
    return {"acknowledgment": ack, "uptake": uptake, "directional_fidelity": fidelity,
            "verdict": "APPROPRIATE_ADAPTATION" if (ack and uptake and fidelity)
            else ("COSMETIC_ADAPTATION" if (ack and not uptake) else "OTHER_FAILURE")}


g_resp = gates(RESPONSIVE)
g_coll = gates(COLLAPSED)
print("responsive:", g_resp)
print("collapsed :", g_coll)

In [ ]:
assert g_resp == {"acknowledgment": True, "uptake": True, "directional_fidelity": True, "verdict": "APPROPRIATE_ADAPTATION"}
assert g_coll["acknowledgment"] is True and g_coll["uptake"] is False
assert g_coll["directional_fidelity"] is False and g_coll["verdict"] == "COSMETIC_ADAPTATION"
print("PASS: keyword mention without decision movement is cosmetic, not reasoning")

## Basin occupancy is marginal; scenario agreement decides

Six distress and six growth scenarios with oracle PRESERVE vs INVEST. Basin occupancy measures concentration; oracle agreement measures correctness. The collapsed stub concentrates perfectly and is still wrong on half the family.

In [ ]:
scenarios = ["DISTRESS"] * 6 + ["GROWTH"] * 6
oracle = ["PRESERVE"] * 6 + ["INVEST"] * 6
resp_dec = ["PRESERVE"] * 6 + ["INVEST"] * 6
coll_dec = ["INVEST"] * 12  # default response basin: one decision whatever the scenario


def basin_occupancy(decs):
    top = max(set(decs), key=decs.count)
    return top, decs.count(top) / len(decs)


def agreement(decs, ora):
    return sum(d == o for d, o in zip(decs, ora)) / len(ora)


resp_top, resp_B = basin_occupancy(resp_dec)
coll_top, coll_B = basin_occupancy(coll_dec)
resp_A = agreement(resp_dec, oracle)
coll_A = agreement(coll_dec, oracle)
oracle_B = 6 / 12
excess = coll_B - oracle_B

print(f"{'stub':12s} {'basin':>8s} {'B_model':>8s} {'agree':>6s}")
print(f"{'responsive':12s} {resp_top:>8s} {resp_B:8.2f} {resp_A:6.2f}")
print(f"{'collapsed':12s} {coll_top:>8s} {coll_B:8.2f} {coll_A:6.2f}")
print(f"excess basin occupancy (collapsed - oracle): {excess:+.2f}")

In [ ]:
assert coll_B == 1.0 and coll_A == 0.5
assert resp_A == 1.0
assert abs(excess - 0.5) < 1e-9
print("PASS: perfect concentration with chance-level agreement is generic collapse")

## What we earned

Static checks can crown the vaguest answer while paired decisive-fact intervention, acknowledgment-vs-uptake gates, and scenario-level agreement expose the default response basin. Low output diversity alone was never the charge; the oracle-required divergence was.

Notebook 11 / Chapter 11 resolves the remaining ambiguity: when thin evidence, not ignored evidence, is the cause, the right behavior is knowing when not to answer.